In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [43]:
import os
import jax
import tyro
import mediapy
import functools
import numpy as np
import mujoco
import mediapy as media

from pathlib import Path
from dataclasses import dataclass
from utils.networks import load_params

import jax.numpy as jnp

In [3]:
AGENT = "crl"

In [4]:
if AGENT == "ppo":
    from ppo import Args as ALGArgs
    @dataclass
    class Args(ALGArgs):
        folder_path: str = "checkpoints/"
        fps: int = 10
        num_envs: int = 1

elif AGENT == "ppo_rnd":
    from ppo_rnd import Args as ALGArgs
    @dataclass
    class Args(ALGArgs):
        folder_path: str = "checkpoints/"
        fps: int = 10
        num_envs: int = 1

elif AGENT == "crl":
    from crl import Args as ALGArgs
    @dataclass
    class Args(ALGArgs):
        folder_path: str = "checkpoints/"
        fps: int = 10
        num_envs: int = 1

else:
    raise NotImplementedError

In [5]:
args = Args()

In [8]:
args.env_id = 'sparse-planar-position-4-cube-3'

In [103]:
args.seed = 76
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [104]:
from builderbench.env_utils import make_env
from utils.wrapper import wrap_env
env_class, default_config = make_env(args)
env = env_class( config=default_config)
action_size = env.action_size

from utils.evaluation import get_video, Evaluator

from crl import Actor, G_encoder, SA_encoder
actor = Actor(action_size=action_size)
g_encoder = G_encoder(rep_size=args.rep_size)
sa_encoder = SA_encoder(rep_size=args.rep_size)

from crl import make_inference_fn
make_policy = make_inference_fn(actor, g_encoder)

In [105]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [106]:
params = load_params(f"../checkpoints/ctrl-in-state__sparse-planar-position-4-cube-3__1__crl__1764520192/params_50.pkl")

In [107]:
actor_params, critic_params = params

jit_inference_fn = jax.jit(
                make_policy(
                    {
                        "actor": actor_params, 
                        "g_encoder": critic_params["g_encoder"]
                    },
                    deterministic=True,
                )
            )

In [108]:
@jax.jit
def critic_grad_ascent(params, state, action, goal):

    actor_params, critic_params = params
    g_repr = g_encoder.apply(critic_params["g_encoder"], goal)
    
    def critic(action):
        sa_repr = sa_encoder.apply(critic_params["sa_encoder"], state, action)
        qf_pi = -jnp.sqrt(jnp.sum((sa_repr - g_repr) ** 2, axis=-1))
        return qf_pi

    critic_grad = jax.grad(critic)
    
    for _ in range(5):
        action = action + 0.01 * critic_grad(action)
        action = jnp.clip(action, -1.0, 1.0)

    return action

In [118]:
rollout = []
returns = []
env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(default_config.episode_length):

    key_policy, sample_key, key = jax.random.split(key, 3)
    target_goal = env_state.info["target_goal"] + jax.random.normal(sample_key, shape=env_state.info["target_goal"].shape) * 0.01
    action, _ = jit_inference_fn(env_state.obs, target_goal, key_policy) 
    action = critic_grad_ascent(params, env_state.obs, action, target_goal)
    print(action)
    
    env_state = step_fn(env_state, action)
    rollout.append(env_state)

    returns.append( env_state.reward )

[ 0.79589224  1.         -0.15981336]
[ 0.68110096  1.         -0.39067233]
[ 0.44592732  1.         -0.6936625 ]
[ 0.06202299  1.         -0.43883583]
[0.6186993  0.10409336 0.07282669]
[ 1.         -0.82095766  0.43511543]
[0.98275363 1.         0.4256393 ]
[0.9641777 0.9853823 0.3615929]
[ 0.9834147  -1.          0.36409476]
[0.88283455 1.         0.2933906 ]
[0.80315953 0.9869778  0.35305843]
[ 9.4316822e-01 -9.7420400e-01  6.3598016e-04]
[ 0.73199856  1.         -0.14866124]
[-0.0391583   1.         -0.62943524]
[ 0.38849705  0.9614497  -0.636326  ]
[-0.52986395  0.8804526  -0.6011569 ]
[-0.19557649  0.8921787  -0.7380536 ]
[-0.02626483 -0.5898248   0.24141194]
[ 0.44944838 -0.84505606 -0.0048871 ]
[ 0.7255677  -0.27506688  0.04821978]
[ 0.05109136  0.2720945  -0.05582061]
[ 0.46806377  0.14154972 -0.058224  ]
[ 0.35688588  1.         -0.03875947]
[-0.06184029 -1.          0.18686576]
[-2.3071858e-01 -6.8927258e-02  1.5938198e-04]
[ 0.26462796 -0.29476056 -0.02732372]
[-0.0981111 

In [119]:
video_images = []
mocap_key = 'target_mocap'
for i in range(default_config.episode_length):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
            )
        )

In [120]:
media.show_video(video_images, fps=10)

This browser does not support the video tag.


In [64]:
env_state.obs[:6]

Array([ 0.01227395, -0.04889322,  0.08050988,  0.0502706 ,  0.10092859,
        0.01034808], dtype=float32)

In [65]:
env_state.obs[12:18]

Array([ 0.01227395, -0.04889321,  0.08046262,  0.04842742,  0.10097638,
        0.01219121], dtype=float32)

In [66]:
env_state.info['target_goal']

Array([ 0.063, -0.063,  0.105,  0.063,  0.147,  0.063],      dtype=float32, weak_type=True)

In [87]:
sample_key, key = jax.random.split(key, 2)
env_state.info["target_goal"] + jax.random.normal(sample_key, shape=env_state.info["target_goal"].shape) * 0.01

Array([ 0.04663356, -0.0598983 ,  0.10858644,  0.06689094,  0.1405803 ,
        0.05414733], dtype=float32)